# House Price Prediction Example

This notebook demonstrates the end-to-end machine learning workflow for
the Kaggle House Prices regression task without requiring a running server.

- Loads (or generates) the dataset via `template_utils`.
- Trains a PyCaret regression model and compares multiple algorithms.
- Finalizes and saves the best model to `ml_model/`.
- Runs direct in-process predictions using the saved model.
- Produces feature importance and neighbourhood price charts.
- Reference: (house_price.example.md)

Follow the reference to write notebooks in a clear manner:
https://github.com/causify-ai/helpers/blob/master/docs/coding/all.jupyter_notebook.how_to_guide.md

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## Imports

In [ ]:
import logging

import helpers.hdbg as hdbg
import helpers.hnotebook as hnotebo
import class_project.project_template.template_utils as cpptteut

## Configuration

In [ ]:
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

hnotebo.config_notebook()

## Load data

Attempt to load the Kaggle CSV; fall back to a synthetic dataset if the
file is absent so the notebook runs without Kaggle credentials.

In [ ]:
# Load or generate the House Prices dataset.
DATA_PATH = "ml_model/train.csv"
df = cpptteut.load_data(DATA_PATH)
_LOG.info("Dataset shape: %s", df.shape)
df.head()

## Compute stats

Inspect the raw data before any cleaning to understand distributions and
identify potential issues.

In [ ]:
# Display summary statistics for the target and key numeric features.
print("Target column statistics:")
print(df[cpptteut.TARGET_COLUMN].describe())
print(f"\nMissing values per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

## Split data

In [ ]:
# Split into train and test sets for offline evaluation.
X_train, X_test, y_train, y_test = cpptteut.split_data(df)
_LOG.info(
    "Train: %d rows  |  Test: %d rows", len(X_train), len(X_test)
)

## Train model with PyCaret

PyCaret's `compare_models` benchmarks multiple regression algorithms in a
single call and selects the best performer by RMSE.

In [ ]:
# Run the PyCaret regression experiment and retrieve the best model.
best_model = cpptteut.run_pycaret_regression(df, n_select=3, fold=5)
# Pull and display the full leaderboard.
leaderboard = cpptteut.get_model_results()
_LOG.info("PyCaret leaderboard retrieved.")
leaderboard

## Save model

In [ ]:
# Finalize on the full dataset and persist to disk.
cpptteut.finalize_and_save(best_model)
_LOG.info("Model finalized and saved.")

## Run in-process predictions

Load the saved model and predict directly without a running Flask server.

In [ ]:
# Load the saved model artifact.
model = cpptteut.load_model_artifact()
# Predict the price of a representative house.
house = {
    "OverallQual": 7,
    "GrLivArea": 1800,
    "GarageCars": 2,
    "YearBuilt": 2005,
    "Neighborhood": "CollgCr",
}
price = cpptteut.predict_price(house, model=model)
_LOG.info("Predicted price: $%.0f", price)
print(f"Predicted sale price: ${price:,.0f}")

## Validate features

In [ ]:
# Demonstrate validation with an intentionally bad payload.
bad_payload = {"OverallQual": 15, "GrLivArea": -50, "ExterQual": "ZZ"}
errors = cpptteut.validate_features(bad_payload)
print("Validation errors:")
for e in errors:
    print(f"  ✗ {e}")

## Show results

Compare predicted prices across neighbourhoods and plot feature importance.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Build one instance per neighbourhood using default feature values.
neighborhoods = ["OldTown", "BrkSide", "CollgCr", "NWAmes", "NoRidge"]
instances = [
    {**cpptteut.FEATURE_DEFAULTS, "Neighborhood": n} for n in neighborhoods
]
# Predict prices for all neighbourhoods.
prices = [cpptteut.predict_price(inst, model=model) for inst in instances]
result_df = (
    pd.DataFrame({"Neighborhood": neighborhoods, "PredictedPrice": prices})
    .sort_values("PredictedPrice")
)
# Plot the neighbourhood comparison bar chart.
plt.figure(figsize=(8, 4))
plt.barh(result_df["Neighborhood"], result_df["PredictedPrice"] / 1000)
plt.xlabel("Predicted Price ($k)")
plt.title("Predicted Price by Neighbourhood (median feature house)")
plt.tight_layout()
plt.savefig("results/price_by_neighborhood.png", dpi=120)
plt.show()
_LOG.info("Plot saved to results/price_by_neighborhood.png.")
print(result_df.to_string(index=False))